In [1]:
import glob
import inspect
import logging
import os
import warnings
from time import time
from typing import Callable

import h5py
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from scipy.optimize import OptimizeWarning, minimize
from scipy.optimize._numdiff import approx_derivative
from scipy.stats import skew
from statsmodels.robust import scale
from statsmodels.robust.norms import RobustNorm, TukeyBiweight

In [2]:
%matplotlib inline

## Function definitions
#### Baselines

In [3]:
# -----------------------------
# Single exponential decay
# -----------------------------
def single_exp(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Baseline with exponential decay (bleaching).

    Parameters
    ----------
    params : np.ndarray
        Parameter vector [b_inf, b, tau]
    t : np.ndarray
        Timestamps

    Returns
    -------
    np.ndarray
        Baseline signal
    """
    b_inf, b, tau = params
    return b_inf * (1 + b * np.exp(-t / tau))


# -----------------------------
# Double exponential decay
# -----------------------------
def double_exp(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Baseline with biphasic exponential decay (bleaching).

    Parameters
    ----------
    params : np.ndarray
        Parameter vector [b_inf, b_slow, b_fast, t_slow, t_fast]
    t : np.ndarray
        Timestamps

    Returns
    -------
    np.ndarray
        Baseline signal
    """
    b_inf, b_slow, b_fast, t_slow, t_fast = params
    return b_inf * (1 + b_slow * np.exp(-t / t_slow)
                    + b_fast * np.exp(-t / t_fast))


# -----------------------------
# Triphasic decay + brightening
# -----------------------------
def bright(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Baseline with triphasic exponential decay (bleaching) multiplied by
    increasing saturating exponential (brightening).

    Parameters
    ----------
    params : np.ndarray
        Parameter vector
        [b_inf, b_slow, b_fast, b_rapid, b_bright,
         t_slow, t_fast, t_rapid, t_bright]
    t : np.ndarray
        Timestamps

    Returns
    -------
    np.ndarray
        Baseline signal
    """
    b_inf, b_slow, b_fast, b_rapid, b_bright, t_slow, t_fast, t_rapid, t_bright = params

    A = 1 + b_slow * np.exp(-t / t_slow) \
        + b_fast * np.exp(-t / t_fast) \
        + b_rapid * np.exp(-t / t_rapid)
    B = 1 - b_bright * np.exp(-t / t_bright)

    return b_inf * A * B

#### Jacobians

In [4]:
# -----------------------------
# Single exponential Jacobian
# -----------------------------
def single_exp_jac(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Jacobian of single exponential baseline.

    Parameters
    ----------
    params : np.ndarray
        [b_inf, b, tau]
    t : np.ndarray
        Timestamps

    Returns
    -------
    J : np.ndarray, shape (len(t), 3)
        Jacobian matrix d(model)/d(params)
    """
    b_inf, b, tau = params
    E = np.exp(-t / tau)
    bE = b_inf * E
    
    J = np.empty((t.size, 3))
    J[:,0] = 1 + b * E              # d/d b_inf
    J[:,1] = bE                     # d/d b
    J[:,2] = b / tau**2 * (bE * t)  # d/d tau

    return J


# -----------------------------
# Double exponential Jacobian
# -----------------------------
def double_exp_jac(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Jacobian of double exponential baseline.

    Parameters
    ----------
    params : np.ndarray
        [b_inf, b_slow, b_fast, t_slow, t_fast]
    t : np.ndarray
        Timestamps

    Returns
    -------
    J : np.ndarray, shape (len(t), 5)
        Jacobian matrix d(model)/d(params)
    """
    b_inf, b_slow, b_fast, t_slow, t_fast = params
    E_slow = np.exp(-t / t_slow)
    E_fast = np.exp(-t / t_fast)
    bE_s = b_inf * E_slow
    bE_f = b_inf * E_fast
    
    J = np.empty((t.size, 5))
    J[:,0] = 1 + b_slow * E_slow + b_fast * E_fast  # d/d b_inf
    J[:,1] = bE_s                             # d/d b_slow
    J[:,2] = bE_f                             # d/d b_fast
    J[:,3] = b_slow / t_slow**2 * (bE_s * t)  # d/d t_slow
    J[:,4] = b_fast / t_fast**2 * (bE_f * t)  # d/d t_fast

    return J


# -----------------------------
# Bright (triphasic + brightening) Jacobian
# -----------------------------
def bright_jac(params: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Jacobian of triphasic exponential baseline with brightening.
    Uses temporary arrays to avoid recomputing repeated terms.

    Parameters
    ----------
    params : np.ndarray
        [b_inf, b_slow, b_fast, b_rapid, b_bright,
         t_slow, t_fast, t_rapid, t_bright]
    t : np.ndarray
        Timestamps

    Returns
    -------
    J : np.ndarray, shape (len(t), 9)
        Jacobian matrix d(model)/d(params)
    """
    b_inf, b_slow, b_fast, b_rapid, b_bright, t_slow, t_fast, t_rapid, t_bright = params
    # exponentials
    E_slow   = np.exp(-t / t_slow)
    E_fast   = np.exp(-t / t_fast)
    E_rapid  = np.exp(-t / t_rapid)
    E_bright = np.exp(-t / t_bright)
    # helper terms
    A = 1 + b_slow*E_slow + b_fast*E_fast + b_rapid*E_rapid
    B = 1 - b_bright*E_bright
    # precompute repeated products
    bE_B_slow  = b_inf * E_slow  * B
    bE_B_fast  = b_inf * E_fast  * B
    bE_B_rapid = b_inf * E_rapid * B
    A_Eb       = A * E_bright
    b_inf_A_Eb = b_inf * A_Eb
    # initialize Jacobian
    J = np.empty((t.size, 9))
    # amplitudes
    J[:,0] = A * B                     # d/d b_inf
    J[:,1] = bE_B_slow                 # d/d b_slow
    J[:,2] = bE_B_fast                 # d/d b_fast
    J[:,3] = bE_B_rapid                # d/d b_rapid
    J[:,4] = -b_inf_A_Eb               # d/d b_bright
    # time constants
    J[:,5] = b_slow    / t_slow**2   * bE_B_slow  * t  # d/d t_slow
    J[:,6] = b_fast    / t_fast**2   * bE_B_fast  * t  # d/d t_fast
    J[:,7] = b_rapid   / t_rapid**2  * bE_B_rapid * t  # d/d t_rapid
    J[:,8] = -b_bright / t_bright**2 * b_inf_A_Eb * t  # d/d t_bright

    return J

#### Baselines + Jacobians

In [5]:
def single_exp_with_jac(params: np.ndarray, t: np.ndarray, return_jac: bool = False) -> np.ndarray:
    """
    Baseline with exponential decay (bleaching).

    Parameters
    ----------
    params : np.ndarray
        Parameter vector [b_inf, b, tau].
    t : np.ndarray
        Timestamps
    return_jac : bool, optional
        If True, return Jacobian alongside the model prediction

    Returns
    -------
    y : np.ndarray
        Model prediction
    J : np.ndarray, optional
        Jacobian with respect to parameters (returned if return_jac=True)
    """
    b_inf, b, tau = params
    E = np.exp(-t / tau)
    A = 1 + b * E
    y = b_inf * A
    if not return_jac:
        return y

    bE = b_inf * E

    J = np.empty((t.size, 3))
    J[:,0] = A                      # d/d b_inf
    J[:,1] = bE                     # d/d b
    J[:,2] = b / tau**2 * (bE * t)  # d/d tau

    return y, J

In [6]:
def double_exp_with_jac(params: np.ndarray, t: np.ndarray, return_jac: bool = False):
    """
    Baseline with biphasic exponential decay (bleaching).

    Parameters
    ----------
    params : np.ndarray
        Parameter vector: [b_inf, b_slow, b_fast, b_rapid, b_bright,
                           t_slow, t_fast, t_rapid, t_bright]
    t : np.ndarray
        Timestamps
    return_jac : bool, optional
        If True, return Jacobian alongside the model prediction

    Returns
    -------
    y : np.ndarray
        Model prediction
    J : np.ndarray, optional
        Jacobian with respect to parameters (returned if return_jac=True)
    """
    b_inf, b_slow, b_fast, t_slow, t_fast = params
    E_slow = np.exp(-t / t_slow)
    E_fast = np.exp(-t / t_fast)
    A = 1 + b_slow * E_slow + b_fast * E_fast
    y = b_inf * A
    if not return_jac:
        return y

    bE_s = b_inf * E_slow
    bE_f = b_inf * E_fast

    J = np.empty((t.size, 5))
    J[:,0] = A                                # d/d b_inf
    J[:,1] = bE_s                             # d/d b_slow
    J[:,2] = bE_f                             # d/d b_fast
    J[:,3] = b_slow / t_slow**2 * (bE_s * t)  # d/d t_slow
    J[:,4] = b_fast / t_fast**2 * (bE_f * t)  # d/d t_fast

    return y, J

In [7]:
def bright_with_jac(params: np.ndarray, t: np.ndarray, return_jac: bool = False):
    """
    Bright baseline with triphasic decay and saturating brightening.

    Parameters
    ----------
    params : np.ndarray
        Parameter vector: [b_inf, b_slow, b_fast, b_rapid, b_bright,
                           t_slow, t_fast, t_rapid, t_bright]
    t : np.ndarray
        Timestamps
    return_jac : bool, optional
        If True, return Jacobian alongside the model prediction

    Returns
    -------
    y : np.ndarray
        Model prediction
    J : np.ndarray, optional
        Jacobian with respect to parameters (returned if return_jac=True)
    """
    b_inf, b_slow, b_fast, b_rapid, b_bright, t_slow, t_fast, t_rapid, t_bright = params
    # exponentials
    E_slow = np.exp(-t / t_slow)
    E_fast = np.exp(-t / t_fast)
    E_rapid = np.exp(-t / t_rapid)
    E_bright = np.exp(-t / t_bright)
    # helper terms
    A = 1 + b_slow * E_slow + b_fast * E_fast + b_rapid * E_rapid
    B = 1 - b_bright * E_bright
    
    y = b_inf * A * B
    if not return_jac:
        return y

    # precompute repeated products
    bE_B_slow  = b_inf * E_slow  * B
    bE_B_fast  = b_inf * E_fast  * B
    bE_B_rapid = b_inf * E_rapid * B
    A_Eb       = A * E_bright
    b_inf_A_Eb = b_inf * A_Eb
    # initialize Jacobian
    J = np.empty((t.size, 9))
    # amplitudes
    J[:,0] = A * B                     # d/d b_inf
    J[:,1] = bE_B_slow                 # d/d b_slow
    J[:,2] = bE_B_fast                 # d/d b_fast
    J[:,3] = bE_B_rapid                # d/d b_rapid
    J[:,4] = -b_inf_A_Eb               # d/d b_bright
    # time constants
    J[:,5] = b_slow    / t_slow**2   * bE_B_slow  * t  # d/d t_slow
    J[:,6] = b_fast    / t_fast**2   * bE_B_fast  * t  # d/d t_fast
    J[:,7] = b_rapid   / t_rapid**2  * bE_B_rapid * t  # d/d t_rapid
    J[:,8] = -b_bright / t_bright**2 * b_inf_A_Eb * t  # d/d t_bright

    return y, J

#### Fit

In [8]:
def tc_brightfit(
    trace: np.ndarray,
    t: np.ndarray,
    rss_thresh: tuple[float, float] = (0.98, 0.997),
    M: RobustNorm = TukeyBiweight(3),
    maxiter: int = 5,
    tol: float = 1e-3,
    update_scale: bool = True,
    skewness_factor: float = 1.0,
    plot: bool = False,
) -> tuple[np.ndarray, np.ndarray, "OptimizeResult"]:
    """Fit trace with baseline (bleaching x brightening) using OLS or IRLS."""

    CEND = "\33[0m"
    CBOLD = "\33[1m"
    CRED = "\33[31m"
    CGREEN = "\33[32m"

    T = len(trace)
    Tds = T // 10
    if rss_thresh == "BIC":
        rss_thresh = [Tds ** (-2 / Tds)] * 2
    elif rss_thresh == "AIC":
        rss_thresh = [np.exp(-4 / Tds)] * 2

    def optimize(trace, x0, ds=1, maxiter=20000, weights=1, plot=plot):
        """Optimize parameters for a given model structure."""
        trace_ds = trace[: T // ds * ds].reshape(-1, ds).mean(1)
        optimize_param = ~np.isnan(x0)

        params = np.array([0] * 5 + [np.inf] * 4)

        def objective(params_to_optimize):
            params[optimize_param] = params_to_optimize
            resid = trace_ds - bright(params, t[ds // 2 :: ds][: len(trace_ds)])
            return np.sum(weights * resid**2)

        bounds = np.array(
            [(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)]
        )

        res = minimize(
            objective,
            np.array(x0)[optimize_param],
            bounds=bounds[optimize_param],
            method="Nelder-Mead",
            options={"maxiter": maxiter},
        )

        params[optimize_param] = res.x
        res.params_full = params.copy()

        logging.info(
            f"Cost: {res.fun:.3f}  "
            f"Success: {CGREEN if res.success else CRED} {res.success} {CEND}  "
            f"{res.message}"
        )

        if plot:
            plot_fit(params, trace, t)

        return params, res

    # ---- double exponential fit ----

    x0 = np.array(
        [trace[-1000:].mean(), 0.35, 0.2, np.nan, np.nan, 3600, 240, np.nan, np.nan]
    )

    logging.info(f"{CBOLD}Fit of 10x decimated trace with double-exp{CEND}")
    x2, res2 = optimize(trace, x0, 10)
    cost2 = res2.fun

    if x2[6] > x2[5]:
        x2[[1, 2, 5, 6]] = x2[[2, 1, 6, 5]]

    # ---- brightening fit ----

    x0[~np.isnan(x0)] = x2[~np.isnan(x0)]
    x0[[4, 8]] = 0.1, 2000

    logging.info(f"{CBOLD}Fit of 10x decimated trace with brightening{CEND}")
    xB, resB = optimize(trace, x0, 10, 3000)
    costB = resB.fun

    cost_ratio = costB / cost2
    include_bright = cost_ratio < rss_thresh[0]

    logging.info(
        f"Cost reduction by including brightening is {(cost_ratio-1)*100:.3f}%, "
        f"thus {CBOLD}{'including' if include_bright else 'skipping'}{CEND} brightening term."
    )

    if include_bright:
        x0[~np.isnan(x0)] = xB[~np.isnan(x0)]
    else:
        x0[[4, 8]] = np.nan

    # ---- third exponential ----

    x0[[3, 7]] = 0.1, 50

    logging.info(f"{CBOLD}Fit of 10x decimated trace with triple-exp{CEND}")
    x3, res3 = optimize(trace, x0, 10, 3000)
    cost3 = res3.fun

    order = np.argsort(x3[5:8])[::-1]
    x3[5:8] = x3[5 + order]
    x3[1:4] = x3[1 + order]

    cost_ratio = cost3 / (costB if include_bright else cost2)
    include_3rd = cost_ratio < rss_thresh[1]

    logging.info(
        f"Cost reduction by including 3rd exponential is {(cost_ratio-1)*100:.3f}%, "
        f"thus {CBOLD}{'including' if include_3rd else 'skipping'}{CEND} 3rd exponential term."
    )

    if include_3rd:
        x0[~np.isnan(x0)] = x3[~np.isnan(x0)]
    else:
        x0[[3, 7]] = np.nan

    params = np.array([0] * 5 + [np.inf] * 4)
    params[~np.isnan(x0)] = x0[~np.isnan(x0)]

    logging.info(
        f"Cost on original trace with params obtained on decimated trace is "
        f"{np.sum((trace - bright(params, t)) ** 2):.3f}"
    )

    logging.info(
        f"{CBOLD}Fit of original trace with "
        f"{'triple-exp' if include_3rd else 'double-exp'} "
        f"and {'' if include_bright else 'no '}brightening{CEND}"
    )

    x, res_final = optimize(trace, x0)

    # ---- IRLS stage ----

    if maxiter > 0 and M is not None and res_final.fun > 0:

        f0 = bright(x, t)
        resid = trace - f0
        scl = scale.mad(resid if skewness_factor == 0 else resid[resid < 0], center=0)

        deviance = M(resid / scl).sum()
        iteration = 0
        converged = False

        while not converged:

            iteration += 1

            weights = M.weights(resid / scl)

            x[np.isnan(x0)] = np.nan
            x, res_final = optimize(trace, x, weights=weights, plot=False)

            f0 = bright(x, t)
            resid = trace - f0

            if update_scale:
                scl = scale.mad(
                    resid if skewness_factor == 0 else resid[resid < 0], center=0
                )

            dev_pre = deviance
            deviance = M(resid / scl).sum()

            converged = iteration >= maxiter or np.abs(deviance / dev_pre - 1) < tol

        res_final.irls_iterations = iteration
        res_final.sigma = scl
        res_final.fun = deviance

    baseline = bright(x, t)

    return baseline, res_final

In [9]:
def fit_baseline_trend(
    trace: np.ndarray,
    t: np.ndarray,
    model: Callable,
    init_params: np.ndarray,
    bounds: tuple[tuple[float, float], ...] | None = None,
    jac: Callable | None = None,
    M: RobustNorm | None = None,
    optimizer: str = "L-BFGS-B",
    maxiter: int = 5,
    tol: float = 1e-3,
    scale_estimator: str = "mad",
    optimizer_options: dict | None = None,
):
    """
    General nonlinear model fitting.

    Parameters
    ----------
    trace : array
        Observed signal
    t : array
        Time vector (passed to model)
    model : callable
        model(params, t) -> model prediction
    init_params : array
        Initial parameter vector
    bounds : array
        Parameter bounds
    jac : callable, optional
        Jacobian  jac(params, t) -> array_like (len(t), n_params)
        If None, gradient is computed numerically by optimizer.
    M : None or RobustNorm object with .rho(u)
        If None -> OLS
        If provided -> IRLS with M-estimator
    optimizer : str
        Method for scipy.optimize.minimize
    maxiter : int
        Maximum IRLS iterations (ignored if M=None)
    tol : float
        Convergence tolerance
    scale_estimator : "mad" or "std"
        Method for scale in IRLS
    optimizer_options : dict
        Options passed to scipy.optimize.minimize

    Returns
    -------
    fitted : np.ndarray
        Fitted model
    res : OptimizeResult
        Optimizer Result
    """

    if optimizer_options is None:
        optimizer_options = {"maxiter": 20000}

    x = np.asarray(init_params).copy()

    # ----------------------------
    # OLS case
    # ----------------------------
    if M is None:

        def objective(p):
            resid = trace - model(p, t)
            loss = np.sum(resid ** 2)
            # loss = np.mean(resid ** 2)
            if jac is None:
                return loss
            J = jac(p, t)
            grad = -2 * J.T @ resid
            # grad = -2/resid.size * J.T @ resid
            return loss, grad

        res = minimize(
            objective,
            x,
            bounds=bounds,
            method=optimizer,
            jac=(jac is not None),
            options=optimizer_options,
        )

        return model(res.x, t), res

    # ----------------------------
    # Robust IRLS case
    # ----------------------------
    for iteration in range(maxiter):

        resid = trace - model(x, t)

        if scale_estimator == "mad":
            sigma = scale.mad(resid, center=0)
            if sigma == 0:
                sigma = np.std(resid)
        else:
            sigma = np.std(resid)

        def objective(p):
            resid = trace - model(p, t)
            u = resid / sigma
            loss = np.sum(M.rho(u))
            # loss = np.mean(M.rho(u))
            if jac is None:
                return loss
            J = jac(p, t)
            psi = M.psi(u)
            grad = -(J.T @ psi) / sigma
            # grad = -(J.T @ psi) / (sigma * resid.size)
            return loss, grad

        res = minimize(
            objective,
            x,
            bounds=bounds,
            method=optimizer,
            jac=(jac is not None),
            options=optimizer_options,
        )

        if np.linalg.norm(res.x - x) / (np.linalg.norm(x) + 1e-12) < tol:
            x = res.x
            break
        x = res.x
        
    res.sigma = sigma

    return model(x, t), res

In [10]:
def fit_baseline_trend2(
    trace: np.ndarray,
    t: np.ndarray,
    model: callable,
    init_params: np.ndarray,
    bounds: tuple[tuple[float, float], ...] | None = None,
    M: RobustNorm | None = None,
    optimizer: str = "L-BFGS-B",
    maxiter: int = 5,
    tol: float = 1e-3,
    scale_estimator: str = "mad",
    optimizer_options: dict | None = None,
):
    """
    Fit a nonlinear baseline to a single trace.
    Supports OLS (M=None) or IRLS (M=M-estimator).
    Model can optionally return a Jacobian as `model(params, t, return_jac=True)`.

    Parameters
    ----------
    trace : np.ndarray
        Observed signal
    t : np.ndarray
        Time vector
    model : callable
        model(params, t) -> prediction
        optionally supports `return_jac=True`
    init_params : np.ndarray
        Initial parameter vector
    bounds : tuple[tuple[float, float], ...] | None
        Parameter bounds
    M : None or RobustNorm
        If None -> OLS, otherwise IRLS with M-estimator
    optimizer : str
        Optimization method for scipy.minimize
    maxiter : int
        Maximum IRLS iterations
    tol : float
        Convergence tolerance
    scale_estimator : str
        "mad" or "std" for IRLS scale
    optimizer_options : dict | None
        Options for scipy.minimize

    Returns
    -------
    fitted : np.ndarray
        Fitted model
    res : OptimizeResult
        Optimizer Result
    """
    if optimizer_options is None:
        optimizer_options = {"maxiter": 20000}

    x = np.asarray(init_params).copy()

    # ----------------------------
    # check if model supports return_jac argument
    sig = inspect.signature(model)
    has_return_jac = "return_jac" in sig.parameters

    # ----------------------------
    # OLS case
    # ----------------------------
    if M is None:

        def objective(p):
            if has_return_jac:
                y_pred, J = model(p, t, return_jac=True)
                resid = trace - y_pred
                grad = -2 * J.T @ resid
                return np.sum(resid**2), grad
            else:
                resid = trace - model(p, t)
                return np.sum(resid**2)

        res = minimize(
            objective,
            x,
            bounds=bounds,
            method=optimizer,
            jac=has_return_jac,
            options=optimizer_options,
        )
        return model(res.x, t), res

    # ----------------------------
    # Robust IRLS case
    # ----------------------------
    for iteration in range(maxiter):

        resid = trace - model(x, t)

        # estimate scale
        if scale_estimator == "mad":
            sigma = scale.mad(resid, center=0)
            if sigma == 0:
                sigma = np.std(resid)
        else:
            sigma = np.std(resid)

        def objective(p):
            r = trace - model(p, t)
            u = r / sigma
            loss = np.sum(M.rho(u))
            if has_return_jac:
                _, J = model(p, t, return_jac=True)
                psi = M.psi(u)
                grad = -(J.T @ psi) / sigma
                return loss, grad
            else:
                return loss

        res = minimize(
            objective,
            x,
            bounds=bounds,
            method=optimizer,
            jac=has_return_jac,
            options=optimizer_options,
        )

        # check convergence
        if np.linalg.norm(res.x - x) / (np.linalg.norm(x) + 1e-12) < tol:
            x = res.x
            break
        x = res.x

    res.sigma = sigma

    return model(x, t), res

#### Fit using JAX

In [11]:
class TukeyBiweight_jax:
    """Tukey biweight M-estimator"""
    def __init__(self, c=4.685):
        self.c = c

    def rho(self, u):
        c = self.c
        abs_u = jnp.abs(u)
        mask = abs_u <= c
        return jnp.where(mask, (c**2/6) * (1 - (1 - (u/c)**2)**3), c**2/6)

    def psi(self, u):
        c = self.c
        mask = jnp.abs(u) <= c
        return jnp.where(mask, u * (1 - (u/c)**2)**2, 0.0)


def single_exp_jax(params, t):
    b_inf, b, tau = params
    return b_inf * (1 + b*jnp.exp(-t/tau))


def double_exp_jax(params, t):
    b_inf, b_slow, b_fast, t_slow, t_fast = params
    return b_inf * (1 + b_slow*jnp.exp(-t/t_slow) + b_fast*jnp.exp(-t/t_fast))


def bright_jax(params, t):
    b_inf, b_slow, b_fast, b_rapid, b_bright, t_slow, t_fast, t_rapid, t_bright = params
    A = 1 + b_slow*jnp.exp(-t/t_slow) + b_fast*jnp.exp(-t/t_fast) + b_rapid*jnp.exp(-t/t_rapid)
    B = 1 - b_bright*jnp.exp(-t/t_bright)
    return b_inf * A * B


def fit_baseline_jax(
    trace: np.ndarray,
    t: np.ndarray,
    model: callable,
    init_params: np.ndarray,
    bounds: tuple[tuple[float, float], ...] | None = None,
    M: TukeyBiweight_jax | None = None,
    optimizer: str = "L-BFGS-B",
    maxiter: int = 5,
    tol: float = 1e-3,
    optimizer_options: dict | None = dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
    dtype=jnp.float64,
):
    """
    JAX-based nonlinear baseline fitting with optional IRLS robust regression.
    """

    # convert data to JAX arrays
    t = jnp.asarray(t, dtype=dtype)
    y = jnp.asarray(trace, dtype=dtype)
    x = jnp.asarray(init_params, dtype=dtype)

    # -------------------------
    # Helper: create fun/jac closure
    # -------------------------
    def make_fun_and_jac(loss_grad_fn):
        grad_cache = {}

        def fun(p):
            val, grad = loss_grad_fn(jnp.asarray(p, dtype=dtype))
            grad_cache["grad"] = np.array(grad)
            return float(val)

        def jac(p):
            return grad_cache["grad"]

        return fun, jac

    # -------------------------
    # Pre-jit loss functions
    # -------------------------
    def ols_loss(params):
        resid = y - model(params, t)
        return jnp.sum(resid ** 2)

    ols_loss_grad = jax.jit(jax.value_and_grad(ols_loss))

    if M is not None:
        def robust_loss(params, sigma):
            resid = y - model(params, t)
            u = resid / sigma
            return jnp.sum(M.rho(u))
        robust_loss_grad = jax.jit(jax.value_and_grad(robust_loss))

    # -------------------------
    # OLS case
    # -------------------------
    if M is None:
        fun, jac = make_fun_and_jac(ols_loss_grad)
        res = minimize(fun, x, jac=jac, bounds=bounds, method=optimizer, options=optimizer_options)
        params = jnp.array(res.x, dtype=dtype)
        fitted = model(params, t)
        return jnp.array(fitted), res

    # -------------------------
    # IRLS loop for robust regression
    # -------------------------
    for iteration in range(maxiter):
        # compute MAD scale
        resid = y - model(x, t)
        sigma = jnp.median(jnp.abs(resid)) * dtype(1.4826)
        sigma = jnp.where(sigma == 0, jnp.std(resid), sigma)

        # make fun/jac for this sigma
        fun, jac = make_fun_and_jac(lambda p: robust_loss_grad(jnp.asarray(p, dtype=dtype), sigma))

        # optimize step
        res = minimize(fun, x, jac=jac, bounds=bounds, method=optimizer, options=optimizer_options)
        x_new = jnp.asarray(res.x, dtype=dtype)

        # update sigma with new params
        resid_new = y - model(x_new, t)
        sigma = jnp.median(jnp.abs(resid_new)) * dtype(1.4826)
        sigma = jnp.where(sigma == 0, jnp.std(resid_new), sigma)

        # check convergence
        eps = dtype(1e-12)
        if jnp.linalg.norm(x_new - x) / (jnp.linalg.norm(x) + eps) < tol:
            x = x_new
            break

        x = x_new

    res.sigma = sigma
    fitted = model(x, t)
    return jnp.array(fitted), res

#### load data 

In [12]:
def get_valid_corrected_f(path):
    dr = path.split("/")[-1]
    data = []
    with h5py.File(f"{path}/{dr}_data.h5") as f:
        for k in f['planes'].keys():
            data.append(f[f'planes/{k}/corrected_f'][f[f'planes/{k}/valid_roi_inds'][:]])
    return data

## Data 

In [13]:
dirs = glob.glob("/data/test-dataset-for-dff_multiplane-ophys_02/*/*")
dirs

['/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/804670',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/775682',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/782149',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753562',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/729088',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/758265',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753561',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/724567',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/755212',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/759075',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/726433',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/747443',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/757436',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/69

In [14]:
traces = get_valid_corrected_f(dirs[0])

In [15]:
[t.shape for t in traces]

[(56, 48374),
 (46, 48374),
 (73, 48374),
 (73, 48374),
 (62, 48374),
 (56, 48374),
 (41, 48374),
 (49, 48374),
 (59, 48403),
 (46, 48403),
 (71, 48403),
 (75, 48403),
 (65, 48403),
 (57, 48403),
 (39, 48403),
 (53, 48403),
 (57, 48427),
 (48, 48427),
 (67, 48427),
 (73, 48427),
 (65, 48427),
 (61, 48427),
 (41, 48427),
 (48, 48427),
 (55, 48390),
 (46, 48390),
 (65, 48390),
 (76, 48390),
 (63, 48390),
 (56, 48390),
 (40, 48390),
 (48, 48390),
 (60, 48373),
 (42, 48373),
 (65, 48373),
 (76, 48373),
 (66, 48373),
 (58, 48373),
 (40, 48373),
 (52, 48373),
 (58, 48366),
 (44, 48366),
 (66, 48366),
 (75, 48366),
 (61, 48366),
 (54, 48366),
 (42, 48366),
 (51, 48366)]

In [16]:
trace = traces[0][0]

In [17]:
frame_rate = 10.63  # looked up manually from session.json of multiplane-ophys_804670_2025-09-24_09-30-56_processed_2025-10-10_22-28-44
timestamps = np.arange(len(trace)) / frame_rate

In [18]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

single_exp_bounds = [(0, np.inf)] * 2 + [(300, np.inf)]
double_exp_bounds = [(0, np.inf)] * 3 + [(300, np.inf), (1, 1200)]
bright_bounds = [(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)]

## check analytical Jacobians 

In [19]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
    (bright, bright_jac, bright_init),
):
    J_num = approx_derivative(lambda p: model(p, timestamps), start_params)
    J_ana = jac(start_params, timestamps)
    print(np.max(np.abs(J_num - J_ana)))  # should be ~1e-8

3.843035756290192e-08
4.878393156104721e-08
5.2436689657042734e-08


In [20]:
for model, jac, start_params in (
    (single_exp, single_exp_with_jac, single_exp_init),
    (double_exp, double_exp_with_jac, double_exp_init),
    (bright, bright_with_jac, bright_init),
):
    J_num = approx_derivative(lambda p: model(p, timestamps), start_params)
    J_ana = jac(start_params, timestamps, True)[1]
    print(np.max(np.abs(J_num - J_ana)))  # should be ~1e-8

3.843035756290192e-08
4.878393156104721e-08
5.2436689657042734e-08


## unbounded OLS 

In [21]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
):
    print("\n" + model.__name__)
    for optimizer in ("Nelder-Mead", "BFGS", "L-BFGS-B", "CG", "Newton-CG", "TNC", "SLSQP", "trust-constr"):
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params,
                                          jac=None if optimizer=="Nelder-Mead" else jac,
                                          optimizer=optimizer)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=200):
            # print(f"{optimizer:13}  Time={tic:7.4f}s  Loss={((F0-trace)**2).mean()/2:.2f}", res.x, res.message)
            print(f"{optimizer:13}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
Nelder-Mead    Time= 0.1511s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Optimization terminated successfully.
BFGS           Time= 0.0321s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B       Time= 0.0225s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
CG             Time= 0.5316s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.35e+03] Desired error not necessarily achieved due to precision loss.
Newton-CG      Time= 0.2170s  Loss=36882.06 [ 1.22e+03 -2.46e-01  3.60e+03] Optimization terminated successfully.
TNC            Time= 0.0802s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Converged (|x_n-x_(n-1)| ~= 0)
SLSQP          Time= 0.2730s  Loss=38781.41 [ 1.05e+03 -6.73e-01  3.68e-03] Optimization terminated successfully
trust-constr   Time= 0.1707s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] `xtol` termination condition is satisfied.

double_exp
Neld

In [22]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
    (bright, bright_jac, bright_init),
):
    print("\n" + model.__name__)
    for method in ("BFGS_noJac", "BFGS", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, jac=None if method[-5:]=="noJac" else jac, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
BFGS_noJac      Time= 0.0663s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
BFGS            Time= 0.0303s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B_noJac  Time= 0.0432s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.0213s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
BFGS_noJac      Time= 1.6409s  Loss=30638.93 [1135.14 -100.98  101.39  458.72  453.49] Desired error not necessarily achieved due to precision loss.
BFGS            Time= 2.8728s  Loss=30638.90 [1135.14 -603.    603.4   456.56  455.68] Desired error not necessarily achieved due to precision loss.
L-BFGS-B_noJac  Time= 7.5868s  Loss=30638.93 [1135.15 -101.93  102.33  458.74  453.56] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### compute model and jac together to share intermediate results 

In [23]:
for model, start_params in (
    (single_exp_with_jac, single_exp_init),
    (double_exp_with_jac, double_exp_init),
    (bright_with_jac, bright_init),
):
    print("\n" + model.__name__)
    for method in ("BFGS", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp_with_jac
BFGS            Time= 0.0902s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 0.0183s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
BFGS            Time= 2.4398s  Loss=30638.90 [1135.14 -603.    603.4   456.56  455.68] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time=11.6583s  Loss=30638.90 [ 1135.14 -1182.01  1182.41   456.34   455.9 ] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
BFGS            Time=14.7546s  Loss=29454.06 [ 3.61e+02  1.35e+06 -1.24e+07  2.18e+07  1.00e+00  2.98e+03  1.13e+02  1.38e+02  4.88e+08] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 3.3385s  Loss=29694.05 [ 7.20e+02  2.17e+00  6.62e+04 -6.62e+04  9.41e-01  3.58e+03  1.22e+02  1.22e+02  1.86e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FA

#### JAX autograd

In [24]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params in (
        (single_exp_jax, single_exp_init),
        (double_exp_jax, double_exp_init),
        (bright_jax, bright_init),
    ):
        print("\n" + model.__name__)
        for optimizer in ("BFGS", "L-BFGS-B"):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_jax(trace, timestamps, model, start_params, optimizer=optimizer,
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)



float64

single_exp_jax
BFGS            Time= 0.4825s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 0.1255s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
BFGS            Time= 1.4363s  Loss=30638.90 [1135.14 -497.31  497.71  456.65  455.59] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 5.1771s  Loss=30638.90 [1135.14 -556.26  556.67  456.57  455.62] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
BFGS            Time= 2.8448s  Loss=29465.62 [ 4.22e+02  1.17e+05 -9.32e+05  1.52e+06  1.00e+00  3.07e+03  9.96e+01  1.35e+02  5.20e+07] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 2.7392s  Loss=29564.33 [ 5.44e+02  3.56e+00  8.24e+04 -8.24e+04  6.76e-01  3.57e+03  4.93e+02  4.93e+02  1.82e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

## bounded OLS 

In [25]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("Nelder-Mead", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds,
                                          jac=jac if method=="L-BFGS-B" else None,
                                          optimizer=optimizer,
                                          optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 0.0864s  Loss=38786.56 [ 1049.25     0.   11951.34] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.0203s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
L-BFGS-B        Time= 0.0217s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
Nelder-Mead     Time= 0.4338s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.00e+02 9.70e+01] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.2372s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.2105s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=26.8927s  Loss=29459.96 [7.31e+02 2.40e+03 2.04e+04 1.82e+02 1.00e+00 2.96e+03 1.58e+02 1.00e+00 1.72e+06] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 8.06

#### compute model and jac together to share intermediate results 

In [26]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("L-BFGS-B",):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp_with_jac
L-BFGS-B        Time= 0.0783s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
L-BFGS-B        Time= 0.1299s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
L-BFGS-B        Time= 3.2751s  Loss=29679.59 [6.87e+02 2.52e+00 3.29e+01 5.10e-02 9.50e-01 3.58e+03 1.47e+02 1.29e+02 1.97e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### JAX autograd

In [27]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_jax(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)



float64

single_exp_jax
L-BFGS-B        Time= 0.1928s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.2565s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 5.0535s  Loss=29482.15 [1.78e+01 1.98e+02 1.42e+03 3.33e+02 9.63e-01 4.71e+03 1.61e+02 1.46e+02 3.31e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_jax
L-BFGS-B        Time= 0.2300s  Loss=38786.56 [1049.25    0.   3485.95] ABNORMAL: 

double_exp_jax
L-BFGS-B        Time= 0.2712s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 0.4863s  Loss=31495.18 [1.09e+03 5.70e-01 3.77e+00 2.86e-01 7.60e-01 3.60e+03 2.40e+02 5.00e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


## bounded robust regression (Tukey) 

In [28]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("Nelder-Mead", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds,
                                         M=TukeyBiweight(3),
                                         jac=jac if method=="L-BFGS-B" else None,
                                         optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 0.7790s  Loss=29155.95 [ 1030.81     0.   18031.9 ] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.0746s  Loss=29160.82 [1030.83    0.   3599.02] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
L-BFGS-B        Time= 0.0398s  Loss=29160.84 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
Nelder-Mead     Time= 2.0645s  Loss=27751.63 [1.02e+03 4.72e-15 5.00e-01 1.92e+04 1.03e+02] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 1.7365s  Loss=27753.58 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.5263s  Loss=27753.59 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=53.8094s  Loss=21633.56 [3.61e+02 3.18e+04 1.65e+04 2.82e+05 1.00e+00 2.96e+03 2.65e+02 1.47e+02 1.15e+07] Optimization terminated successfully.
L-BFGS-B_noJac  Time=15.88

#### compute model and jac together to share intermediate results 

In [29]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("L-BFGS-B",):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                          M=TukeyBiweight(3),
                                          optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp_with_jac
L-BFGS-B        Time= 0.1049s  Loss=29160.84 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
L-BFGS-B        Time= 0.6859s  Loss=27753.59 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
L-BFGS-B        Time=13.6043s  Loss=21595.90 [6.96e+01 5.18e+01 8.08e+02 1.41e+03 9.91e-01 4.28e+03 1.40e+02 2.09e+01 3.21e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### JAX autograd

In [30]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_jax(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                           M=TukeyBiweight_jax(3),
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)



float64

single_exp_jax
L-BFGS-B        Time= 0.7471s  Loss=29142.03 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.7027s  Loss=27745.82 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 2.7015s  Loss=21236.60 [6.43e+02 2.77e+00 6.43e+01 2.43e+02 9.94e-01 3.53e+03 1.30e+02 1.70e+01 1.89e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_jax
L-BFGS-B        Time= 0.7404s  Loss=29154.92 [1030.9     0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.5356s  Loss=28268.69 [1.01e+03 0.00e+00 2.90e-01 3.60e+03 2.23e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 0.6631s  Loss=22573.69 [1.06e+03 5.78e-01 4.31e+00 0.00e+00 7.51e-01 3.60e+03 1.89e+02 5.32e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR

#### tc_bright

In [31]:
tic = -time()
F0, res = tc_brightfit(trace, np.arange(len(trace)) / frame_rate, M=TukeyBiweight(3), skewness_factor=0)
tic += time()
with np.printoptions(precision=2, suppress=False, linewidth=120):
    print(f"{'bright':14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)

bright          Time= 4.1460s  Loss=21887.81 [1.11e+03 2.62e-13 1.55e+01 7.63e+01 9.89e-01 1.24e+04 1.24e+02 1.46e+01 6.88e+02] Optimization terminated successfully.


#### scale params to all be O(1)

In [32]:
def bright_scaled(params, t, b_inf_scale: float = 1000.0, return_jac: bool = False):
    """
    Scaled wrapper for bright model.

    Parameters
    ----------
    params : np.ndarray
        [b_inf_scaled, b_slow, b_fast, b_rapid, b_bright,
         log_t_slow, log_t_fast, log_t_rapid, log_t_bright]
    t : np.ndarray
        Time vector
    b_inf_scale : float
        Scaling factor for b_inf
    return_jac : bool
        If True, return Jacobian in optimizer parameter space
    """

    (
        b_inf_scaled,
        b_slow,
        b_fast,
        b_rapid,
        b_bright,
        log_t_slow,
        log_t_fast,
        log_t_rapid,
        log_t_bright,
    ) = params

    # transform parameters
    b_inf = b_inf_scaled * b_inf_scale
    t_slow = np.exp(log_t_slow)
    t_fast = np.exp(log_t_fast)
    t_rapid = np.exp(log_t_rapid)
    t_bright = np.exp(log_t_bright)

    phys_params = (
        b_inf,
        b_slow,
        b_fast,
        b_rapid,
        b_bright,
        t_slow,
        t_fast,
        t_rapid,
        t_bright,
    )

    # evaluate model
    if not return_jac:
        return bright(phys_params, t)

    y, J_phys = bright_with_jac(phys_params, t, return_jac=True)

    # chain rule transform
    J = np.empty_like(J_phys)

    # amplitudes
    J[:, 0] = J_phys[:, 0] * b_inf_scale
    J[:, 1] = J_phys[:, 1]
    J[:, 2] = J_phys[:, 2]
    J[:, 3] = J_phys[:, 3]
    J[:, 4] = J_phys[:, 4]

    # log-time constants
    J[:, 5] = J_phys[:, 5] * t_slow
    J[:, 6] = J_phys[:, 6] * t_fast
    J[:, 7] = J_phys[:, 7] * t_rapid
    J[:, 8] = J_phys[:, 8] * t_bright

    return y, J


tic = -time()

b_inf0 = trace[-1000:].mean()
b_inf_scale = 1000.0


def fit_baseline_trend_scaled(
    trace: np.ndarray,
    t: np.ndarray,
    model: callable,
    init_params: np.ndarray,
    bounds: tuple[tuple[float, float], ...] | None = None,
    M: RobustNorm | None = None,
    optimizer: str = "L-BFGS-B",
    maxiter: int = 5,
    optimizer_options: dict | None = None,
):

    b_inf_scale = 1000.0
    
    init_params_scaled = init_params.copy()
    init_params_scaled[0] /= b_inf_scale
    init_params_scaled[5:] = np.log(init_params_scaled[5:])
    
    bounds = np.array([(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)])
    bounds_scaled = bounds.copy()
    bounds_scaled[5:] = np.log(bounds_scaled[5:])
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        F0, res = fit_baseline_trend2(
            trace,
            timestamps,
            model=bright_scaled,  # lambda p, t: bright_scaled(np.array(p), t, b_inf_scale=b_inf_scale),
            init_params=init_params_scaled,
            bounds=bounds_scaled,
            M=M,
            optimizer=optimizer,
            maxiter=maxiter,
            optimizer_options=optimizer_options,
        )
    
    res.x[0] *= b_inf_scale        # rescale b_inf
    res.x[5:] = np.exp(res.x[5:])  # log → time constants

    return F0, res

In [33]:
tic = -time()
F0, res = fit_baseline_trend_scaled(trace, timestamps, model, bright_init, bounds=bright_bounds,
                                    M=TukeyBiweight(3),
                                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
tic += time()
with np.printoptions(precision=2, suppress=False, linewidth=120):
    print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)

L-BFGS-B        Time=19.5470s  Loss=21685.56 [3.51e+03 9.71e+01 8.39e+02 1.41e+02 1.00e+00 2.89e+03 1.58e+02 1.84e+01 3.42e+05] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### multiple traces in parallel

In [34]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
    for method in ("L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            res = Parallel(n_jobs=n_jobs)(
                delayed(fit_baseline_trend)(
                    t, timestamps, model, start_params, bounds=bounds,
                    M=TukeyBiweight(3),
                    jac=jac if method=="L-BFGS-B" else None,
                    optimizer=optimizer,
                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
                for t in traces[0]
            )
        tic += time()
        with np.printoptions(precision=5, suppress=False, linewidth=120):
            nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
            print(f"{method:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}")


single_exp
L-BFGS-B_noJac  Time=  6.8259s  Norm.Loss=0.29393
L-BFGS-B        Time= 24.2064s  Norm.Loss=0.29344

double_exp
L-BFGS-B_noJac  Time= 12.7614s  Norm.Loss=0.28991
L-BFGS-B        Time= 20.4826s  Norm.Loss=0.29041

bright
L-BFGS-B_noJac  Time=135.1583s  Norm.Loss=0.23358
L-BFGS-B        Time= 98.2005s  Norm.Loss=0.23338


In [35]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
    for optimizer in ("L-BFGS-B",):
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            res = Parallel(n_jobs=n_jobs)(
                delayed(fit_baseline_trend2)(
                    t, timestamps, model, start_params, bounds=bounds,
                    M=TukeyBiweight(3),
                    optimizer=optimizer,
                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
                for t in traces[0]
            )
        tic += time()
        with np.printoptions(precision=5, suppress=False, linewidth=120):
            nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
            print(f"{method:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}")


single_exp_with_jac
L-BFGS-B        Time= 14.2717s  Norm.Loss=0.29344

double_exp_with_jac
L-BFGS-B        Time= 13.6884s  Norm.Loss=0.29041

bright_with_jac
L-BFGS-B        Time=108.4595s  Norm.Loss=0.23338


In [37]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    
    def fit_baseline_worker(trace):
        # Re-enable float64 in this worker process
        jax.config.update("jax_enable_x64", True)
        # Call your existing function
        return fit_baseline_jax(
            trace,
            timestamps,
            model,
            start_params,
            bounds=bounds,
            M=TukeyBiweight_jax(3),
            optimizer=optimizer,
            optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
            dtype=dtype,
        )

    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                res = Parallel(n_jobs=n_jobs)(
                    delayed(fit_baseline_worker)(t) for t in traces[0]
                )
            tic += time()
            with np.printoptions(precision=5, suppress=False, linewidth=120):
                nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
                print(f"{optimizer:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}") 



float64

single_exp_jax
L-BFGS-B        Time= 18.5066s  Norm.Loss=0.29184

double_exp_jax
L-BFGS-B        Time= 23.9781s  Norm.Loss=0.28916

bright_jax
L-BFGS-B        Time= 57.0088s  Norm.Loss=0.22831


float32

single_exp_jax
L-BFGS-B        Time= 10.2289s  Norm.Loss=0.28812

double_exp_jax
L-BFGS-B        Time= 12.0055s  Norm.Loss=0.28384

bright_jax
L-BFGS-B        Time= 18.1732s  Norm.Loss=0.23156
